<a href="https://colab.research.google.com/github/SANGHATI23/ohdsi-fhir-omop-showcase-demo/blob/main/OHDSI_Original25_EarliestExecution_Evidence_Audit_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Clone the complete repository history

In [1]:
import sys, subprocess, os, json, re, shutil, hashlib
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

REPO_URL = "https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git"
REPO_DIR = Path("/content/ohdsi-fhir-omop-showcase-demo-earlyaudit")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--no-single-branch", REPO_URL, str(REPO_DIR)],
    check=True
)
subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"],
    check=True
)

RESULT_DIR = REPO_DIR / "results" / "original25_earliest_execution_audit"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_DIR)

Repository: /content/ohdsi-fhir-omop-showcase-demo-earlyaudit


## 1. Restrict the analysis to the early project window

The later analytical-stability work was added in July 2026.  
For this audit, we focus on commits **through June 10, 2026**, which includes the initial showcase, mapping analytics, workflow figure, and early Phase-4 outputs but excludes July stability/comparator work and the August reviewer-recovery notebooks.

In [2]:
EARLY_CUTOFF = "2026-06-11T00:00:00Z"

def git(args, binary=False):
    p = subprocess.run(
        ["git", "-C", str(REPO_DIR)] + args,
        capture_output=True,
        text=not binary
    )
    return p

log = git([
    "log", "--all", "--reverse", "--date=iso-strict",
    "--pretty=format:%H%x09%ad%x09%an%x09%s"
])

rows = []
for line in log.stdout.splitlines():
    parts = line.split("\t", 3)
    if len(parts) == 4:
        rows.append({
            "commit": parts[0],
            "date": parts[1],
            "author": parts[2],
            "subject": parts[3],
        })

all_history_df = pd.DataFrame(rows)
all_history_df["date_parsed"] = pd.to_datetime(all_history_df["date"], utc=True, errors="coerce")
cutoff = pd.Timestamp(EARLY_CUTOFF)
early_history_df = all_history_df[all_history_df["date_parsed"] < cutoff].copy()

print("All commits:", len(all_history_df))
print("Early commits through June 10:", len(early_history_df))
display(early_history_df[["commit","date","subject"]])

early_history_df.drop(columns=["date_parsed"]).to_csv(
    RESULT_DIR / "early_commit_history.csv", index=False
)

All commits: 21
Early commits through June 10: 7


,commit,date,subject
0,35314736dd3e5959e53660ae07326c4327e59dbf,2026-05-28T20:16:05-05:00,Initial OHDSI FHIR to OMOP showcase demo
1,07a57280d10591352cd3690409b43740c740a982,2026-06-02T04:39:14-06:00,Add Athena vocabulary mapping analytics
2,11aa90c2632d58a8afda53f30124b9964cf39c80,2026-06-02T04:53:24-06:00,Update README with Athena vocabulary mapping r...
3,69d239187f62f5330257c6e56dfc1c867467960d,2026-06-04T10:25:34-05:00,Add OHDSI showcase workflow figure
4,43131773e371df52121aedcf280f6428ac230cdb,2026-06-04T10:30:41-05:00,Add Athena vocabulary mapping bar plot
5,f08d8917b15b6961842421a4d0494a6aefab000c,2026-06-10T12:41:06+00:00,Add Phase 4 analytical stability outputs
6,d8baf74f259fb35ac108466ea58afbb23052e5a6,2026-06-10T08:16:37-05:00,Revise README for OHDSI FHIR-to-OMOP Showcase ...


# Part A — Executed notebook evidence

## 2. Parse early notebooks structurally

Each notebook cell is classified separately as:

- `EXECUTED_OUTPUT` — the evidence is in a saved output from an executed cell
- `CODE_SOURCE` — code contains the term, but no corresponding output
- `MARKDOWN_SOURCE` — prose only

This prevents manuscript text from being mistaken for measured output.

In [3]:
EARLY_COMMITS = early_history_df["commit"].tolist()

TARGET_TERMS = [
    "25", "27", "1386", "983", "1275", "14168", "14150",
    "78.12", "visit_concept_id", "drug_concept_id",
    "person", "patient"
]

notebook_cells = []

for commit in EARLY_COMMITS:
    date = early_history_df.loc[early_history_df["commit"] == commit, "date"].iloc[0]
    tree = git(["ls-tree", "-r", "--name-only", commit])
    notebooks = [f for f in tree.stdout.splitlines() if f.lower().endswith(".ipynb")]

    for fp in notebooks:
        show = git(["show", f"{commit}:{fp}"])
        if show.returncode != 0:
            continue
        try:
            obj = json.loads(show.stdout)
        except Exception:
            continue

        for idx, cell in enumerate(obj.get("cells", [])):
            source = "".join(cell.get("source", []))
            output_parts = []

            for output in cell.get("outputs", []) or []:
                if "text" in output:
                    val = output["text"]
                    output_parts.append("".join(val) if isinstance(val, list) else str(val))
                data = output.get("data", {})
                for key in ("text/plain", "text/markdown"):
                    if key in data:
                        val = data[key]
                        output_parts.append("".join(val) if isinstance(val, list) else str(val))

            output_text = "\n".join(output_parts)
            source_low = source.lower()
            output_low = output_text.lower()

            source_terms = [t for t in TARGET_TERMS if t.lower() in source_low]
            output_terms = [t for t in TARGET_TERMS if t.lower() in output_low]

            if not source_terms and not output_terms:
                continue

            if output_terms and cell.get("execution_count") is not None:
                evidence_class = "EXECUTED_OUTPUT"
            elif cell.get("cell_type") == "code":
                evidence_class = "CODE_SOURCE"
            else:
                evidence_class = "MARKDOWN_SOURCE"

            notebook_cells.append({
                "commit": commit,
                "date": date,
                "file": fp,
                "cell_index": idx,
                "cell_type": cell.get("cell_type"),
                "execution_count": cell.get("execution_count"),
                "evidence_class": evidence_class,
                "source_terms": " | ".join(source_terms),
                "output_terms": " | ".join(output_terms),
                "source_excerpt": source[:5000],
                "output_excerpt": output_text[:8000],
            })

evidence_df = pd.DataFrame(notebook_cells)
print("Relevant early notebook cells:", len(evidence_df))
if not evidence_df.empty:
    print("\nEvidence-class counts:")
    display(evidence_df["evidence_class"].value_counts().rename_axis("class").reset_index(name="cells"))
    evidence_df.to_csv(
        RESULT_DIR / "early_notebook_cell_evidence.csv", index=False
    )

Relevant early notebook cells: 0


## 3. Find direct executed evidence for 25 source Patients and 27 OMOP PERSON rows

The filters below require the values to appear in **saved executed output**, not merely in code or prose.

In [4]:
executed = (
    evidence_df[evidence_df["evidence_class"] == "EXECUTED_OUTPUT"].copy()
    if not evidence_df.empty else pd.DataFrame()
)

def contains_all(text, terms):
    low = str(text).lower()
    return all(str(t).lower() in low for t in terms)

patient25_rows = []
person27_rows = []
full_signature_rows = []
mapping_rows = []

for _, r in executed.iterrows() if not executed.empty else []:
    out = str(r["output_excerpt"])
    low = out.lower()

    # Require patient-ish context + 25.
    if "25" in low and ("patient" in low or "fhir" in low or "synthea" in low):
        patient25_rows.append(r.to_dict())

    # Require person-ish context + 27.
    if "27" in low and "person" in low:
        person27_rows.append(r.to_dict())

    # At least four of the submitted clinical counts in one output is strong count-signature evidence.
    sig_values = ["1386", "983", "1275", "14168", "14150"]
    sig_count = sum(v in low for v in sig_values)
    if sig_count >= 4:
        d = r.to_dict()
        d["signature_values_present"] = sig_count
        full_signature_rows.append(d)

    if "78.12" in low or (
        "visit_concept_id" in low and ("0.00" in low or "0.0" in low)
    ):
        mapping_rows.append(r.to_dict())

patient25_df = pd.DataFrame(patient25_rows)
person27_df = pd.DataFrame(person27_rows)
signature_df = pd.DataFrame(full_signature_rows)
mapping_exec_df = pd.DataFrame(mapping_rows)

print("Executed cells with 25 + Patient/FHIR context:", len(patient25_df))
print("Executed cells with 27 + PERSON context:", len(person27_df))
print("Executed cells with >=4 submitted clinical counts:", len(signature_df))
print("Executed cells with mapping-gap output:", len(mapping_exec_df))

for name, df in [
    ("patient25_execution_evidence.csv", patient25_df),
    ("person27_execution_evidence.csv", person27_df),
    ("submitted_count_signature_execution_evidence.csv", signature_df),
    ("mapping_gap_execution_evidence.csv", mapping_exec_df),
]:
    if not df.empty:
        df.to_csv(RESULT_DIR / name, index=False)

if not person27_df.empty:
    display(person27_df[[
        "date","file","cell_index","execution_count","output_terms","output_excerpt"
    ]].head(30))

Executed cells with 25 + Patient/FHIR context: 0
Executed cells with 27 + PERSON context: 0
Executed cells with >=4 submitted clinical counts: 0
Executed cells with mapping-gap output: 0


## 4. Link 25 and 27 evidence to the same historical execution context

Evidence is stronger if:

- it occurs in the same notebook revision,
- nearby executed cells reference the same database path,
- the 25 source count precedes the 27 PERSON count,
- both belong to the same commit.

This cell does not assume that same-commit means same-run; it reports the strength of the linkage.

In [5]:
linkage_rows = []

if not patient25_df.empty and not person27_df.empty:
    for _, p25 in patient25_df.iterrows():
        for _, p27 in person27_df.iterrows():
            if p25["commit"] != p27["commit"]:
                continue
            same_file = p25["file"] == p27["file"]
            cell_distance = abs(int(p25["cell_index"]) - int(p27["cell_index"])) if same_file else None
            source_before_target = (
                int(p25["cell_index"]) <= int(p27["cell_index"])
                if same_file else None
            )

            strength = "same_commit_only"
            if same_file:
                strength = "same_notebook_revision"
            if same_file and cell_distance is not None and cell_distance <= 5:
                strength = "same_notebook_nearby_cells"

            linkage_rows.append({
                "commit": p25["commit"],
                "date": p25["date"],
                "patient25_file": p25["file"],
                "patient25_cell": p25["cell_index"],
                "person27_file": p27["file"],
                "person27_cell": p27["cell_index"],
                "same_file": same_file,
                "cell_distance": cell_distance,
                "source_before_target": source_before_target,
                "linkage_strength": strength,
            })

linkage_df = pd.DataFrame(linkage_rows)
print("Potential 25→27 evidence linkages:", len(linkage_df))
if not linkage_df.empty:
    linkage_df = linkage_df.sort_values(
        ["same_file","cell_distance"], ascending=[False, True]
    )
    display(linkage_df.head(50))
    linkage_df.to_csv(
        RESULT_DIR / "patient25_person27_execution_linkage.csv", index=False
    )

Potential 25→27 evidence linkages: 0


# Part B — Historical database lifecycle

## 5. Search early code for the exact SQLite database path and reset/reuse behavior

The relevant question is not whether the word `append` appears somewhere.  
We specifically look for code that:

- defines the SQLite path,
- deletes/unlinks/removes it,
- initializes OMOP tables,
- writes/imports into the same path,
- or explicitly reuses an existing non-empty database.

In [6]:
TEXT_EXTS = (".ipynb", ".py", ".sh", ".md", ".txt")

DB_TERMS = [
    "sqlite", ".db", ".sqlite", "database_url", "db_path",
    "unlink(", "os.remove(", "shutil.rmtree", "drop_all",
    "drop table", "create_all", "create_engine",
    "pyomop", "import", "load"
]

db_code_rows = []

for commit in EARLY_COMMITS:
    date = early_history_df.loc[early_history_df["commit"] == commit, "date"].iloc[0]
    tree = git(["ls-tree", "-r", "--name-only", commit])
    files = [f for f in tree.stdout.splitlines() if f.lower().endswith(TEXT_EXTS)]

    for fp in files:
        show = git(["show", f"{commit}:{fp}"])
        if show.returncode != 0:
            continue

        if fp.lower().endswith(".ipynb"):
            try:
                obj = json.loads(show.stdout)
            except Exception:
                continue
            for cell_idx, cell in enumerate(obj.get("cells", [])):
                if cell.get("cell_type") != "code":
                    continue
                source = "".join(cell.get("source", []))
                low = source.lower()
                matched = [t for t in DB_TERMS if t.lower() in low]
                if matched:
                    db_code_rows.append({
                        "commit": commit,
                        "date": date,
                        "file": fp,
                        "cell_or_line": f"cell:{cell_idx}",
                        "terms": " | ".join(matched),
                        "code_excerpt": source[:8000],
                    })
        else:
            for line_no, line in enumerate(show.stdout.splitlines(), 1):
                low = line.lower()
                matched = [t for t in DB_TERMS if t.lower() in low]
                if matched:
                    db_code_rows.append({
                        "commit": commit,
                        "date": date,
                        "file": fp,
                        "cell_or_line": f"line:{line_no}",
                        "terms": " | ".join(matched),
                        "code_excerpt": line[:4000],
                    })

db_code_df = pd.DataFrame(db_code_rows)
print("Early DB lifecycle/code evidence rows:", len(db_code_df))
if not db_code_df.empty:
    db_code_df.to_csv(
        RESULT_DIR / "early_database_lifecycle_code_evidence.csv", index=False
    )

# High-specificity reset evidence only.
reset_pattern = re.compile(
    r"(unlink\s*\(|os\.remove\s*\(|Path\([^)]*\)\.unlink|drop_all|DROP\s+TABLE|rm\s+-f\s+.*\.(db|sqlite))",
    re.IGNORECASE
)
reset_df = (
    db_code_df[db_code_df["code_excerpt"].str.contains(reset_pattern, regex=True, na=False)].copy()
    if not db_code_df.empty else pd.DataFrame()
)

print("High-specificity database reset/destruction evidence rows:", len(reset_df))
if not reset_df.empty:
    display(reset_df[["date","file","cell_or_line","terms","code_excerpt"]].head(50))
    reset_df.to_csv(
        RESULT_DIR / "database_reset_evidence.csv", index=False
    )

Early DB lifecycle/code evidence rows: 543
High-specificity database reset/destruction evidence rows: 0


/tmp/ipykernel_8048/3477745310.py:69: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  db_code_df[db_code_df["code_excerpt"].str.contains(reset_pattern, regex=True, na=False)].copy()


## 6. Search for saved outputs indicating a pre-existing/non-empty database

A saved runtime message about an existing database is stronger evidence than generic lifecycle code.

In [7]:
reuse_rows = []

if not executed.empty:
    for _, r in executed.iterrows():
        out = str(r["output_excerpt"])
        low = out.lower()
        phrases = [
            "non-empty database",
            "database already exists",
            "already exists",
            "existing database",
            "reusing database",
            "found database",
            "database files found",
        ]
        matched = [p for p in phrases if p in low]
        if matched:
            d = r.to_dict()
            d["reuse_phrases"] = " | ".join(matched)
            reuse_rows.append(d)

reuse_df = pd.DataFrame(reuse_rows)
print("Early executed outputs suggesting database reuse/existence:", len(reuse_df))
if not reuse_df.empty:
    display(reuse_df[[
        "date","file","cell_index","execution_count","reuse_phrases","output_excerpt"
    ]].head(50))
    reuse_df.to_csv(
        RESULT_DIR / "database_reuse_execution_evidence.csv", index=False
    )

Early executed outputs suggesting database reuse/existence: 0


# Part C — Mapping-gap evidence

## 7. Isolate early executed outputs for mapping completeness

This extracts only saved execution output involving:

- `drug_exposure`
- `visit_occurrence`
- `drug_concept_id`
- `visit_concept_id`
- mapping percentages/counts

In [8]:
mapping_context_rows = []

if not executed.empty:
    for _, r in executed.iterrows():
        out = str(r["output_excerpt"])
        low = out.lower()
        if (
            ("drug_exposure" in low or "drug_concept_id" in low or
             "visit_occurrence" in low or "visit_concept_id" in low)
            and
            ("mapped" in low or "mapping" in low or "%" in out or "percent" in low)
        ):
            mapping_context_rows.append(r.to_dict())

mapping_context_df = pd.DataFrame(mapping_context_rows)
print("Early executed mapping-context cells:", len(mapping_context_df))
if not mapping_context_df.empty:
    display(mapping_context_df[[
        "date","file","cell_index","execution_count","output_excerpt"
    ]].head(50))
    mapping_context_df.to_csv(
        RESULT_DIR / "early_mapping_execution_context.csv", index=False
    )

Early executed mapping-context cells: 0


## 8. Search early code/prose for an explicit explanation of drug/visit mapping gaps

A true explanation should mention a cause such as source coding, vocabulary coverage, transformation logic, or encounter class mapping—not merely restate the percentages.

In [9]:
CAUSE_TERMS = [
    "because", "cause", "reason", "unmapped", "source code", "coding system",
    "vocabulary", "mapping logic", "encounter class", "visit class",
    "medication", "rxnorm", "snomed", "concept_id", "source_concept"
]

cause_rows = []

for commit in EARLY_COMMITS:
    date = early_history_df.loc[early_history_df["commit"] == commit, "date"].iloc[0]
    tree = git(["ls-tree", "-r", "--name-only", commit])
    files = [
        f for f in tree.stdout.splitlines()
        if f.lower().endswith((".ipynb",".md",".py",".txt",".csv"))
    ]

    for fp in files:
        show = git(["show", f"{commit}:{fp}"])
        if show.returncode != 0:
            continue
        text = show.stdout
        low = text.lower()

        has_domain = any(x in low for x in [
            "drug_exposure", "drug_concept_id",
            "visit_occurrence", "visit_concept_id"
        ])
        if not has_domain:
            continue

        for line_no, line in enumerate(text.splitlines(), 1):
            ll = line.lower()
            if not any(x in ll for x in [
                "drug_exposure", "drug_concept_id",
                "visit_occurrence", "visit_concept_id",
                "78.12", "0.00"
            ]):
                continue
            matched_causes = [t for t in CAUSE_TERMS if t in ll]
            cause_rows.append({
                "commit": commit,
                "date": date,
                "file": fp,
                "line": line_no,
                "cause_terms": " | ".join(matched_causes),
                "text": line[:5000],
            })

cause_df = pd.DataFrame(cause_rows)
print("Early domain-specific mapping evidence lines:", len(cause_df))
if not cause_df.empty:
    display(cause_df.head(100))
    cause_df.to_csv(
        RESULT_DIR / "early_mapping_cause_candidate_lines.csv", index=False
    )

Early domain-specific mapping evidence lines: 273


,commit,date,file,line,cause_terms,text
0,35314736dd3e5959e53660ae07326c4327e59dbf,2026-05-28T20:16:05-05:00,README.md,59,,│ ├── 05_drug_exposure_summary.py
1,35314736dd3e5959e53660ae07326c4327e59dbf,2026-05-28T20:16:05-05:00,outputs/queries/ohdsi_demo_asset_checklist.md,28,,- visit_occurrence count
2,35314736dd3e5959e53660ae07326c4327e59dbf,2026-05-28T20:16:05-05:00,outputs/queries/ohdsi_demo_asset_checklist.md,30,,- drug_exposure count
3,35314736dd3e5959e53660ae07326c4327e59dbf,2026-05-28T20:16:05-05:00,outputs/queries/ohdsi_demo_asset_checklist.md,86,,outputs/queries/drug_exposure_source_values_to...
4,35314736dd3e5959e53660ae07326c4327e59dbf,2026-05-28T20:16:05-05:00,outputs/queries/ohdsi_demo_narration_script.md,31,,"Using pyOMOP, the FHIR NDJSON files are import..."
...,...,...,...,...,...,...
95,69d239187f62f5330257c6e56dfc1c867467960d,2026-06-04T10:25:34-05:00,outputs/queries/ohdsi_showcase_summary.md,18,,| drug_exposure | 1275 |
96,69d239187f62f5330257c6e56dfc1c867467960d,2026-06-04T10:25:34-05:00,outputs/queries/standard_drug_concepts_top20.csv,1,vocabulary | concept_id,"concept_name,domain_id,vocabulary_id,drug_conc..."
97,69d239187f62f5330257c6e56dfc1c867467960d,2026-06-04T10:25:34-05:00,outputs/queries/table_counts.csv,4,,"visit_occurrence,1386"
98,69d239187f62f5330257c6e56dfc1c867467960d,2026-06-04T10:25:34-05:00,outputs/queries/table_counts.csv,5,,"drug_exposure,1275"


# Part D — Evidence grading

## 9. Grade what the preserved early history actually supports

The grade is intentionally conservative.

In [10]:
has_25_exec = len(patient25_df) > 0
has_27_exec = len(person27_df) > 0
has_same_notebook_link = (
    (not linkage_df.empty) and
    (linkage_df["same_file"] == True).any()
)
has_full_count_signature = len(signature_df) > 0
has_mapping_exec = len(mapping_context_df) > 0

# Reset evidence is useful only if it is in the same early workflow files; this remains evidence, not proof.
has_reset_code = len(reset_df) > 0
has_reuse_exec = len(reuse_df) > 0

if has_25_exec and has_27_exec and has_same_notebook_link and has_full_count_signature:
    count_evidence_grade = "STRONG_HISTORICAL_EXECUTION_EVIDENCE"
elif has_27_exec and has_full_count_signature:
    count_evidence_grade = "MODERATE_HISTORICAL_EXECUTION_EVIDENCE"
else:
    count_evidence_grade = "INSUFFICIENT_EXECUTION_EVIDENCE"

# Causality requires more than counts.
if has_reuse_exec and has_reset_code:
    cause_grade = "AMBIGUOUS_DATABASE_LIFECYCLE_EVIDENCE"
else:
    cause_grade = "NO_DIRECT_CAUSAL_EVIDENCE"

grading_df = pd.DataFrame([{
    "executed_25_patient_evidence": has_25_exec,
    "executed_27_person_evidence": has_27_exec,
    "same_notebook_25_27_link": has_same_notebook_link,
    "full_submitted_count_signature_execution": has_full_count_signature,
    "mapping_execution_evidence": has_mapping_exec,
    "database_reset_code_evidence": has_reset_code,
    "database_reuse_runtime_evidence": has_reuse_exec,
    "count_evidence_grade": count_evidence_grade,
    "cause_evidence_grade": cause_grade,
}])

display(grading_df)
grading_df.to_csv(
    RESULT_DIR / "early_evidence_grading.csv", index=False
)

,executed_25_patient_evidence,executed_27_person_evidence,same_notebook_25_27_link,full_submitted_count_signature_execution,mapping_execution_evidence,database_reset_code_evidence,database_reuse_runtime_evidence,count_evidence_grade,cause_evidence_grade
0,False,False,False,False,False,False,False,INSUFFICIENT_EXECUTION_EVIDENCE,NO_DIRECT_CAUSAL_EVIDENCE


## 10. Produce a reviewer-safe interpretation

This output is phrased for **decision-making**, not automatically for copy/paste into a manuscript.

In [11]:
if count_evidence_grade == "STRONG_HISTORICAL_EXECUTION_EVIDENCE" and cause_grade == "NO_DIRECT_CAUSAL_EVIDENCE":
    person_interpretation = (
        "The early repository history preserves strong execution evidence for the submitted small-cohort "
        "counts, but it does not preserve sufficient row-level lineage to establish why 25 source Patients "
        "corresponded to 27 OMOP PERSON rows. The discrepancy is historically verified but causally unresolved."
    )
elif count_evidence_grade == "STRONG_HISTORICAL_EXECUTION_EVIDENCE":
    person_interpretation = (
        "The early repository history preserves strong execution evidence for the small-cohort counts and "
        "some database-lifecycle signals, but the lifecycle evidence is not sufficient by itself to establish "
        "a causal explanation for the 25-to-27 discrepancy."
    )
else:
    person_interpretation = (
        "The preserved early repository history is insufficient to establish both the source 25-patient count "
        "and the 27-row OMOP PERSON result as one directly traceable run. A clean reproduction is required."
    )

mapping_interpretation = (
    "Historical mapping outputs were recovered."
    if has_mapping_exec else
    "No sufficiently specific early executed mapping output was recovered."
)

decision = (
    "CLEAN_REPRODUCTION_RECOMMENDED"
    if cause_grade != "DIRECT_CAUSE_ESTABLISHED"
    else "HISTORICAL_CAUSE_ESTABLISHED"
)

decision_df = pd.DataFrame([{
    "person_interpretation": person_interpretation,
    "mapping_interpretation": mapping_interpretation,
    "next_scientific_action": decision,
    "manuscript_rule": (
        "Do not name stale rows, duplicate import, Synthea behavior, or pyOMOP behavior as the cause "
        "unless a later direct test establishes it."
    )
}])

display(decision_df)
decision_df.to_csv(
    RESULT_DIR / "reviewer_safe_interpretation.csv", index=False
)

print("\nPERSON INTERPRETATION:\n", person_interpretation)
print("\nNEXT SCIENTIFIC ACTION:", decision)

,person_interpretation,mapping_interpretation,next_scientific_action,manuscript_rule
0,The preserved early repository history is insu...,No sufficiently specific early executed mappin...,CLEAN_REPRODUCTION_RECOMMENDED,"Do not name stale rows, duplicate import, Synt..."



PERSON INTERPRETATION:
 The preserved early repository history is insufficient to establish both the source 25-patient count and the 27-row OMOP PERSON result as one directly traceable run. A clean reproduction is required.

NEXT SCIENTIFIC ACTION: CLEAN_REPRODUCTION_RECOMMENDED


## 11. Generate compact audit report

In [12]:
report_lines = [
    "# OHDSI Original 25-Patient Earliest Execution Evidence Audit",
    "",
    f"- Early commits audited: **{len(early_history_df)}**",
    f"- Executed 25-Patient evidence cells: **{len(patient25_df)}**",
    f"- Executed 27-PERSON evidence cells: **{len(person27_df)}**",
    f"- Full submitted count-signature execution cells: **{len(signature_df)}**",
    f"- Mapping execution-context cells: **{len(mapping_context_df)}**",
    f"- High-specificity DB-reset code rows: **{len(reset_df)}**",
    f"- Runtime DB-reuse/existence evidence cells: **{len(reuse_df)}**",
    "",
    f"**Count evidence grade:** {count_evidence_grade}",
    "",
    f"**Cause evidence grade:** {cause_grade}",
    "",
    "## Interpretation",
    "",
    person_interpretation,
    "",
    "## Mapping",
    "",
    mapping_interpretation,
    "",
    "## Next action",
    "",
    f"**{decision}**",
    "",
    "The next reproduction must use a fresh database, a fixed 25-patient source cohort, explicit source-to-PERSON "
    "reconciliation, mapping-gap diagnostics, and stage-level timing.",
]

report_path = RESULT_DIR / "OHDSI_ORIGINAL25_EARLY_EXECUTION_AUDIT.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print(report_path.read_text(encoding="utf-8"))

# OHDSI Original 25-Patient Earliest Execution Evidence Audit

- Early commits audited: **7**
- Executed 25-Patient evidence cells: **0**
- Executed 27-PERSON evidence cells: **0**
- Full submitted count-signature execution cells: **0**
- Mapping execution-context cells: **0**
- High-specificity DB-reset code rows: **0**
- Runtime DB-reuse/existence evidence cells: **0**

**Count evidence grade:** INSUFFICIENT_EXECUTION_EVIDENCE

**Cause evidence grade:** NO_DIRECT_CAUSAL_EVIDENCE

## Interpretation

The preserved early repository history is insufficient to establish both the source 25-patient count and the 27-row OMOP PERSON result as one directly traceable run. A clean reproduction is required.

## Mapping

No sufficiently specific early executed mapping output was recovered.

## Next action

**CLEAN_REPRODUCTION_RECOMMENDED**

The next reproduction must use a fresh database, a fixed 25-patient source cohort, explicit source-to-PERSON reconciliation, mapping-gap diagnostics, and stag

## 12. Public outputs

This notebook creates only GitHub-safe evidence tables under:

```text
results/original25_earliest_execution_audit/
```

No raw FHIR data, OMOP database, Athena vocabulary, or patient identifiers are copied.

### What to do after running

Save the executed notebook to GitHub and share the link.

If the final decision is **`CLEAN_REPRODUCTION_RECOMMENDED`**, the next notebook will be the actual reviewer-resolution run:

**fresh fixed 25-patient FHIR cohort → fresh OMOP DB → exact 25→PERSON reconciliation → drug/visit root-cause diagnostics → timing → automated tests.**